In [8]:
import pandas as pd
from pathlib import Path

# =========================
# 1. 文件路径
# =========================
base_dir = Path("/Users/littlestars/Desktop/grain_project/data/raw")   # 如果在本地跑，改成你的文件夹路径
file_ag_org = base_dir / "t2_501_2.xls"
file_farm = base_dir / "t2_501_8.xls"
file_household = base_dir / "t2_501_17.xls"

# =========================
# 2. 通用读取函数
# =========================
def read_grain_area_table(file_path, owner_type):
    """
    读取类似 t2_501_2.xls / t2_501_8.xls / t2_501_17.xls
    输出长表：
    region | year | area | owner_type
    """
    df = pd.read_excel(file_path, header=None)

    # 不同表的表头起始行略有不同：
    # _2 和 _8：年份在第5行(索引4)
    # _17：年份在第4行(索引3)
    year_row = None
    for i in range(min(10, len(df))):
        vals = df.iloc[i, 1:12].tolist()
        # 判断这一行是不是年份行：大部分是 2007~2017
        numeric_vals = pd.to_numeric(pd.Series(vals), errors="coerce")
        if numeric_vals.notna().sum() >= 8:
            year_row = i
            break

    if year_row is None:
        raise ValueError(f"未找到年份行: {file_path}")

    years = pd.to_numeric(df.iloc[year_row, 1:12], errors="coerce").astype("Int64").tolist()

    # 数据从年份行下一行开始
    data = df.iloc[year_row + 1:, [0] + list(range(1, 12))].copy()
    data.columns = ["region"] + years

    # 去掉空行
    data = data[data["region"].notna()].copy()

    # 去掉完全空的数据行
    data = data.dropna(how="all", subset=years)

    # 转成长表
    long_df = data.melt(
        id_vars="region",
        value_vars=years,
        var_name="year",
        value_name="area"
    )

    # 数值化
    long_df["area"] = pd.to_numeric(long_df["area"], errors="coerce")
    long_df["year"] = pd.to_numeric(long_df["year"], errors="coerce").astype("Int64")
    long_df["owner_type"] = owner_type

    return long_df

# =========================
# 3. 分别读取三张表
# =========================
df_ag = read_grain_area_table(file_ag_org, "agricultural_organizations")
df_farm = read_grain_area_table(file_farm, "farmers_individual_entrepreneurs")
df_house = read_grain_area_table(file_household, "households")

# =========================
# 4. 改成宽表并合并
# =========================
df_ag = df_ag.rename(columns={"area": "area_ag_org"})[["region", "year", "area_ag_org"]]
df_farm = df_farm.rename(columns={"area": "area_farm"})[["region", "year", "area_farm"]]
df_house = df_house.rename(columns={"area": "area_household"})[["region", "year", "area_household"]]

panel = df_ag.merge(df_farm, on=["region", "year"], how="outer")
panel = panel.merge(df_house, on=["region", "year"], how="outer")

# =========================
# 5. 计算总面积
# =========================
panel["area_total"] = (
    panel["area_ag_org"].fillna(0)
    + panel["area_farm"].fillna(0)
    + panel["area_household"].fillna(0)
)

# =========================
# 6. 简单增加地区层级标签（可选）
# =========================
def classify_level(region_name):
    if region_name == "Российская Федерация":
        return "country"
    elif "Федеральный округ" in str(region_name):
        return "federal_district"
    else:
        return "region"

panel["level"] = panel["region"].apply(classify_level)

# =========================
# 7. 排序
# =========================
panel = panel.sort_values(["level", "region", "year"]).reset_index(drop=True)

# =========================
# 8. 保存结果
# =========================
out_csv = base_dir / "grain_area_panel_reproduced.csv"
out_xlsx = base_dir / "grain_area_panel_reproduced.xlsx"

panel.to_csv(out_csv, index=False, encoding="utf-8-sig")
panel.to_excel(out_xlsx, index=False)

print("已生成：")
print(out_csv)
print(out_xlsx)

print("\n前20行预览：")
print(panel.head(20))

已生成：
/Users/littlestars/Desktop/grain_project/data/raw/grain_area_panel_reproduced.csv
/Users/littlestars/Desktop/grain_project/data/raw/grain_area_panel_reproduced.xlsx

前20行预览：
                                region  year  area_ag_org  area_farm  \
0                 Российская Федерация  2007    33753.518  10111.445   
1                 Российская Федерация  2008    35362.534  10951.720   
2                 Российская Федерация  2009    35713.270  11377.325   
3                 Российская Федерация  2010    32048.187  10669.504   
4                 Российская Федерация  2011    32113.568  10959.958   
5                 Российская Федерация  2012    32120.076  11806.796   
6                 Российская Федерация  2013    32643.520  12750.926   
7                 Российская Федерация  2014    32147.140  13514.844   
8                 Российская Федерация  2015    32051.874  14096.366   
9                 Российская Федерация  2016    31932.646  14686.566   
10                Российская 

In [12]:
!pip install -U openpyxl

Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [openpyxl]1/2 [openpyxl]


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

/opt/anaconda3/lib/python3.12/site-packages/pandas/core/computation/expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/opt/anaconda3/lib/python3.12/site-packages/pandas/core/arrays/masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


In [9]:
pip install openpyxl>=3.1.5

zsh:1: 3.1.5 not found
Note: you may need to restart the kernel to use updated packages.


In [7]:
import sys
!{sys.executable} -m pip install --upgrade "openpyxl>=3.1.5"

Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple
  Using cached https://pypi.tuna.tsinghua.edu.cn/packages/c0/da/977ded879c29cbd04de313843e76868e6e13408a94ed6b987245dc7c8506/openpyxl-3.1.5-py2.py3-none-any.whl (250 kB)
  Attempting uninstall: openpyxl
    Found existing installation: openpyxl 3.1.2
    Uninstalling openpyxl-3.1.2:
      Successfully uninstalled openpyxl-3.1.2


In [7]:
import pandas as pd
import numpy as np
from pathlib import Path

base = Path("/Users/littlestars/Desktop/grain_project/data/raw")   # 本地运行时改成你的文件夹

# =========================
# 1. 读取 2007-2017 原表
# =========================
old_panel = pd.read_excel(base / "grain_area_panel_reproduced.xlsx")

# 只保留我们真正要的总面积列
old_panel = old_panel.copy()
old_panel["region"] = old_panel["region"].astype(str).str.strip()
old_panel["year"] = pd.to_numeric(old_panel["year"], errors="coerce").astype("Int64")

# 如果原表已有 area_total，就直接用
old_panel_final = old_panel[["region", "year", "area_total"]].copy()

# =========================
# 2. 读取 2018-2023 六个 sheet
#    这六个 sheet 的 F 列就是总面积
# =========================
xlsx_file = base / "grain_area_2018_2023.xlsx"
xl = pd.ExcelFile(xlsx_file)

records = []

for sheet in xl.sheet_names:
    year = int(sheet)   # sheet 名假设就是 2018, 2019, ..., 2023
    df = pd.read_excel(xlsx_file, sheet_name=sheet, header=None)

    # 找数据起始行：A列有地区名，F列有数值
    start_row = None
    for i in range(len(df)):
        region = df.iloc[i, 0] if df.shape[1] > 0 else np.nan
        value_f = df.iloc[i, 5] if df.shape[1] > 5 else np.nan

        if pd.notna(region) and pd.to_numeric(pd.Series([value_f]), errors="coerce").notna().iloc[0]:
            start_row = i
            break

    if start_row is None:
        continue

    temp = df.iloc[start_row:, [0, 5]].copy()
    temp.columns = ["region", "area_total"]

    temp["region"] = temp["region"].astype(str).str.strip()
    temp["year"] = year
    temp["area_total"] = pd.to_numeric(temp["area_total"], errors="coerce")

    # 去掉空行和非数据行
    temp = temp[temp["region"].notna()].copy()
    temp = temp[temp["area_total"].notna()].copy()

    records.append(temp[["region", "year", "area_total"]])

new_panel = pd.concat(records, ignore_index=True)

# =========================
# 3. 地区层级标签（可选）
# =========================
def classify_level(region_name):
    if region_name == "Российская Федерация":
        return "country"
    elif "Федеральный округ" in str(region_name):
        return "federal_district"
    else:
        return "region"

old_panel_final["level"] = old_panel_final["region"].apply(classify_level)
new_panel["level"] = new_panel["region"].apply(classify_level)

# =========================
# 4. 纵向拼接
# =========================
final_panel = pd.concat([old_panel_final, new_panel], ignore_index=True)

# 去重：如果同一个 region-year 重复，保留最后一个
final_panel = final_panel.drop_duplicates(subset=["region", "year"], keep="last")

# 排序
final_panel = final_panel.sort_values(["region", "year"]).reset_index(drop=True)

# =========================
# 5. 保存
# =========================
final_panel.to_excel(base / "grain_area_panel_2007_2023_final.xlsx", index=False)
final_panel.to_csv(base / "grain_area_panel_2007_2023_final.csv", index=False, encoding="utf-8-sig")

print("完成。")
print(final_panel.head(20))
print(final_panel.tail(20))

完成。
              region  year  area_total   level
0                  A  2022     5.00000  region
1                  A  2023     5.00000  region
2     Алтайский край  2007  3576.55300  region
3     Алтайский край  2008  3781.66600  region
4     Алтайский край  2009  3803.96800  region
5     Алтайский край  2010  3393.56400  region
6     Алтайский край  2011  3628.32200  region
7     Алтайский край  2012  3538.06200  region
8     Алтайский край  2013  3536.19200  region
9     Алтайский край  2014  3717.01000  region
10    Алтайский край  2015  3632.10100  region
11    Алтайский край  2016  3646.22500  region
12    Алтайский край  2017  3746.32400  region
13    Алтайский край  2018  3240.38200  region
14    Алтайский край  2019  3169.04300  region
15    Алтайский край  2020  3289.34000  region
16    Алтайский край  2021  3234.64900  region
17    Алтайский край  2022  3353.13131  region
18    Алтайский край  2023  3352.99945  region
19  Амурская область  2007   231.97500  region
         

In [9]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

# =========================================================
# 0. 路径
# =========================================================
base = Path("/Users/littlestars/Desktop/grain_project/data/raw") 
input_file = base / "grain_area_panel_2007_2023_final.xlsx"

output_xlsx = base / "grain_area_panel_2007_2023_cleaned.xlsx"
output_csv = base / "grain_area_panel_2007_2023_cleaned.csv"

# =========================================================
# 1. 读取数据
# =========================================================
df = pd.read_excel(input_file)

print("原始数据形状:", df.shape)
print(df.head())

# =========================================================
# 2. 基础标准化
# =========================================================
df = df.copy()

# 列名统一
df.columns = [str(c).strip() for c in df.columns]

# 保证关键列存在
required_cols = ["region", "year", "area_total"]
for col in required_cols:
    if col not in df.columns:
        raise ValueError(f"缺少必要列: {col}")

# region 转字符串并去空格
df["region"] = df["region"].astype(str).str.strip()

# 去掉连续空白
df["region"] = df["region"].str.replace(r"\s+", " ", regex=True)

# year / area_total 数值化
df["year"] = pd.to_numeric(df["year"], errors="coerce")
df["area_total"] = pd.to_numeric(df["area_total"], errors="coerce")

# 删掉 year 或 area_total 缺失的行
df = df.dropna(subset=["year", "area_total"]).copy()
df["year"] = df["year"].astype(int)

# =========================================================
# 3. 名称统一函数
# =========================================================
def normalize_region_name(name: str) -> str:
    if pd.isna(name):
        return name
    
    x = str(name).strip()
    x = re.sub(r"\s+", " ", x)

    # 全大写转正常首字母形式的一些常见情况不强行 title，
    # 避免俄文专名被误伤，只做定向替换
    replace_map = {
        "РОССИЙСКАЯ ФЕДЕРАЦИЯ": "Российская Федерация",
        "ЦЕНТРАЛЬНЫЙ ФЕДЕРАЛЬНЫЙ ОКРУГ": "Центральный федеральный округ",
        "СЕВЕРО-ЗАПАДНЫЙ ФЕДЕРАЛЬНЫЙ ОКРУГ": "Северо-Западный федеральный округ",
        "ЮЖНЫЙ ФЕДЕРАЛЬНЫЙ ОКРУГ": "Южный федеральный округ",
        "СЕВЕРО-КАВКАЗСКИЙ ФЕДЕРАЛЬНЫЙ ОКРУГ": "Северо-Кавказский федеральный округ",
        "ПРИВОЛЖСКИЙ ФЕДЕРАЛЬНЫЙ ОКРУГ": "Приволжский федеральный округ",
        "УРАЛЬСКИЙ ФЕДЕРАЛЬНЫЙ ОКРУГ": "Уральский федеральный округ",
        "СИБИРСКИЙ ФЕДЕРАЛЬНЫЙ ОКРУГ": "Сибирский федеральный округ",
        "ДАЛЬНЕВОСТОЧНЫЙ ФЕДЕРАЛЬНЫЙ ОКРУГ": "Дальневосточный федеральный округ",
        "КРЫМСКИЙ ФЕДЕРАЛЬНЫЙ ОКРУГ": "Крымский федеральный округ",
        "ЕВРЕЙСКАЯ АВТОНОМНАЯ ОБЛАСТЬ": "Еврейская автономная область",
        "РЕСПУБЛИКА АДЫГЕЯ": "Республика Адыгея",
        "РЕСПУБЛИКА БАШКОРТОСТАН": "Республика Башкортостан",
        "РЕСПУБЛИКА БУРЯТИЯ": "Республика Бурятия",
        "РЕСПУБЛИКА ДАГЕСТАН": "Республика Дагестан",
        "РЕСПУБЛИКА ИНГУШЕТИЯ": "Республика Ингушетия",
        "КАБАРДИНО-БАЛКАРСКАЯ РЕСПУБЛИКА": "Кабардино-Балкарская Республика",
        "РЕСПУБЛИКА КАЛМЫКИЯ": "Республика Калмыкия",
        "КАРАЧАЕВО-ЧЕРКЕССКАЯ РЕСПУБЛИКА": "Карачаево-Черкесская Республика",
        "РЕСПУБЛИКА КАРЕЛИЯ": "Республика Карелия",
        "РЕСПУБЛИКА КОМИ": "Республика Коми",
        "РЕСПУБЛИКА МАРИЙ ЭЛ": "Республика Марий Эл",
        "РЕСПУБЛИКА МОРДОВИЯ": "Республика Мордовия",
        "РЕСПУБЛИКА САХА (ЯКУТИЯ)": "Республика Саха (Якутия)",
        "РЕСПУБЛИКА СЕВЕРНАЯ ОСЕТИЯ-АЛАНИЯ": "Республика Северная Осетия-Алания",
        "РЕСПУБЛИКА ТАТАРСТАН": "Республика Татарстан",
        "РЕСПУБЛИКА ТЫВА": "Республика Тыва",
        "УДМУРТСКАЯ РЕСПУБЛИКА": "Удмуртская Республика",
        "РЕСПУБЛИКА ХАКАСИЯ": "Республика Хакасия",
        "ЧЕЧЕНСКАЯ РЕСПУБЛИКА": "Чеченская Республика",
        "ЧУВАШСКАЯ РЕСПУБЛИКА": "Чувашская Республика",
        "АЛТАЙСКИЙ КРАЙ": "Алтайский край",
        "ЗАБАЙКАЛЬСКИЙ КРАЙ": "Забайкальский край",
        "КАМЧАТСКИЙ КРАЙ": "Камчатский край",
        "КРАСНОДАРСКИЙ КРАЙ": "Краснодарский край",
        "КРАСНОЯРСКИЙ КРАЙ": "Красноярский край",
        "ПЕРМСКИЙ КРАЙ": "Пермский край",
        "ПРИМОРСКИЙ КРАЙ": "Приморский край",
        "СТАВРОПОЛЬСКИЙ КРАЙ": "Ставропольский край",
        "ХАБАРОВСКИЙ КРАЙ": "Хабаровский край",
    }

    if x in replace_map:
        x = replace_map[x]

    # 常见细节统一
    small_replace = {
        "г.Севастополь": "г. Севастополь",
        "г.Москва": "г. Москва",
        "г.Санкт-Петербург": "г. Санкт-Петербург",
        "Еврейская Автономная область": "Еврейская автономная область",
        "Чукотский автономный округ ": "Чукотский автономный округ",
        "Ханты-Мансийский автономный округ-Югра": "Ханты-Мансийский автономный округ - Югра",
        "Ямало-Hенецкий автономный округ": "Ямало-Ненецкий автономный округ",
        "Ханты-Мансийский автономный округ - Югра ": "Ханты-Мансийский автономный округ - Югра",
    }
    x = small_replace.get(x, x)

    return x

# 记录原始名称与新名称映射
rename_df = df[["region"]].drop_duplicates().copy()
rename_df["region_cleaned"] = rename_df["region"].apply(normalize_region_name)

df["region"] = df["region"].apply(normalize_region_name)

# =========================================================
# 4. 识别要删除的行
# =========================================================
def is_bad_row(region: str) -> bool:
    if pd.isna(region):
        return True

    r = str(region).strip()

    # 明显脏行
    if r in {"A", "nan", "", "-"}:
        return True

    # 全国总计
    if r == "Российская Федерация":
        return True

    # 联邦区
    r_low = r.lower()
    if "федеральный округ" in r_low:
        return True

    # 非标准主体的小计行 / 注释行
    bad_patterns = [
        "без автономного округа",
        "без автономных округов",
        "в т.ч.",
        "в том числе",
    ]
    for pat in bad_patterns:
        if pat in r_low:
            return True

    return False

df["drop_flag"] = df["region"].apply(is_bad_row)

dropped_rows = df[df["drop_flag"]].copy()
clean_df = df[~df["drop_flag"]].copy()

# =========================================================
# 5. 再做一轮面板质量检查
# =========================================================
# 去掉 area_total 小于 0 的异常
clean_df = clean_df[clean_df["area_total"] >= 0].copy()

# 检查重复 region-year
dup_mask = clean_df.duplicated(subset=["region", "year"], keep=False)
duplicates = clean_df[dup_mask].sort_values(["region", "year"]).copy()

# 如果重复，默认按 region-year 聚合取最大值或均值都可以
# 这里更稳妥：如果同组 area_total 完全一致，保留一条；
# 否则取均值并标记
if len(duplicates) > 0:
    agg_df = (
        clean_df.groupby(["region", "year"], as_index=False)
        .agg(area_total=("area_total", "mean"))
    )
    clean_df = agg_df.copy()
else:
    clean_df = clean_df.drop_duplicates(subset=["region", "year"]).copy()

# =========================================================
# 6. 地区层级标签
# =========================================================
def classify_region_type(region_name: str) -> str:
    r = str(region_name)

    if "Республика" in r:
        return "republic"
    elif "край" in r:
        return "krai"
    elif "область" in r:
        return "oblast"
    elif "автономный округ" in r:
        return "autonomous_okrug"
    elif "автономная область" in r:
        return "autonomous_oblast"
    elif r.startswith("г. "):
        return "federal_city"
    else:
        return "other"

clean_df["region_type"] = clean_df["region"].apply(classify_region_type)

# =========================================================
# 7. 排序
# =========================================================
clean_df = clean_df.sort_values(["region", "year"]).reset_index(drop=True)

# =========================================================
# 8. 汇总信息
# =========================================================
summary = pd.DataFrame({
    "item": [
        "raw_rows",
        "clean_rows",
        "dropped_rows",
        "unique_regions_clean",
        "year_min",
        "year_max",
        "duplicate_region_year_rows_before_fix"
    ],
    "value": [
        len(df),
        len(clean_df),
        len(dropped_rows),
        clean_df["region"].nunique(),
        clean_df["year"].min(),
        clean_df["year"].max(),
        len(duplicates)
    ]
})

notes = pd.DataFrame({
    "rule": [
        "删除全国总计",
        "删除联邦区汇总",
        "删除脏行",
        "删除非标准小计行",
        "统一地区名称",
        "删除重复或聚合重复 region-year"
    ],
    "detail": [
        "Российская Федерация",
        "包含 'федеральный округ'",
        "如 A, 空字符串, nan",
        "如 'без автономного округа', 'в т.ч.'",
        "统一大小写、空格和常见写法",
        "若存在重复 region-year，则按均值聚合"
    ]
})

# =========================================================
# 9. 输出
# =========================================================
clean_df.to_csv(output_csv, index=False, encoding="utf-8-sig")

with pd.ExcelWriter(output_xlsx, engine="openpyxl") as writer:
    clean_df.to_excel(writer, sheet_name="clean_panel", index=False)
    dropped_rows.to_excel(writer, sheet_name="dropped_rows", index=False)
    rename_df.to_excel(writer, sheet_name="rename_map", index=False)
    summary.to_excel(writer, sheet_name="summary", index=False)
    notes.to_excel(writer, sheet_name="notes", index=False)
    if len(duplicates) > 0:
        duplicates.to_excel(writer, sheet_name="duplicates_before_fix", index=False)

print("清洗完成。")
print("输出文件：")
print(output_xlsx)
print(output_csv)

print("\n清洗摘要：")
print(summary)

print("\n清洗后前10行：")
print(clean_df.head(10))

print("\n清洗后后10行：")
print(clean_df.tail(10))

原始数据形状: (1509, 4)
           region  year  area_total   level
0               A  2022       5.000  region
1               A  2023       5.000  region
2  Алтайский край  2007    3576.553  region
3  Алтайский край  2008    3781.666  region
4  Алтайский край  2009    3803.968  region
清洗完成。
输出文件：
/Users/littlestars/Desktop/grain_project/data/raw/grain_area_panel_2007_2023_cleaned.xlsx
/Users/littlestars/Desktop/grain_project/data/raw/grain_area_panel_2007_2023_cleaned.csv

清洗摘要：
                                    item  value
0                               raw_rows   1509
1                             clean_rows   1323
2                           dropped_rows    186
3                   unique_regions_clean     84
4                               year_min   2007
5                               year_max   2023
6  duplicate_region_year_rows_before_fix      0

清洗后前10行：
           region  year  area_total   level  drop_flag region_type
0  Алтайский край  2007    3576.553  region      False     